# Graphagate Training & Evaluation

Questo notebook permette di addestrare e testare il modello **Temporal Graph Network** (TGN) in locale, sfruttando l'accelerazione **Metal Performance Shaders (MPS)** dei Mac.
In questo modo non c'è bisogno di usare Docker e si possono ottenere prestazioni di training nettamente superiori sul chip M4 Pro.

In [1]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import sys

if torch.backends.mps.is_available():
    print("MPS accelerato (Metal) trovato. Verrà utilizzata la GPU del Mac.")
elif torch.cuda.is_available():
    print("CUDA GPU trovata.")
else:
    print("GPU non trovata, si userà la CPU.")

MPS accelerato (Metal) trovato. Verrà utilizzata la GPU del Mac.


## 1. Configurazione Iperparametri
Qui modifichiamo i parametri per riflettere il nuovo scaling deciso nel plan (es. 1000 utenti + 1000 guest, 2000 device).

In [2]:
from graphagate.config import TGNConfig
from graphagate.train_tgn import train_tgn

cfg = TGNConfig(
    # Scale defaults up as discussed
    num_users=1000,
    num_devices=2000,
    num_sources=1500,
    num_configs=400,
    num_events=50000,   # Increase events since we have more entities
    
    # Training params
    epochs=20,           # Keep low for initial testing
    batch_size=200,
    eval_batch_size=200,
    
    # Ensure memory capacity accounts for the extra 1000 guests
    capacity_headroom=2000
)

## 2. Addestramento e Valutazione
Eseguiamo il processo completo. La pipeline includerà la generazione dei dati sintetici aggiornati (con ritmi circadiani e lateral movement chains).

In [ ]:
# Patch temporanea per il bug di PyTorch MPS con int64 su scatter_reduce_
import torch
import torch_geometric.nn.models.tgn as tgn

_original_scatter = tgn.scatter
def _patched_scatter(src, index, dim=0, dim_size=None, reduce='sum'):
    if reduce in ['max', 'amax'] and src.dtype == torch.int64 and getattr(src.device, 'type', '') == 'mps':
        return _original_scatter(src.float(), index, dim, dim_size, reduce).long()
    return _original_scatter(src, index, dim, dim_size, reduce)
tgn.scatter = _patched_scatter

# Esegui il training. Verranno salvati gli artefatti in public/
metrics = train_tgn(cfg)

print("\n--- Training Completato ---")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")